In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "HuggingFaceTB/SmolLM-135M"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

text = "What is the capital of France?"
inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=20, use_cache=False)
tokenizer.decode(outputs, skip_special_tokens=True)[0]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

'What is the capital of France?\nParis, the capital of France, is the largest city in the country and the second-largest'

In [2]:
import math
from transformers.models.llama.modeling_llama import LlamaAttention

# B = batch size
# S = seq length
# H = hidden size
# N_q = number of query heads
# N_kv = number of kv heads
# D = head dim
# G = number of query heads per kv head
def forward(
    self,
    hidden_states: torch.Tensor,  # (B, S, H)
    position_embeddings: tuple[torch.Tensor, torch.Tensor] = None,  # ((B, S, D), (B, S, D))
    attention_mask: torch.Tensor = None,  # (B, 1, S, S)
    output_attentions: bool = False,
    **kwargs,
) -> tuple[torch.Tensor, torch.Tensor | None]:
    # self.q_proj: (H) -> (N_q * D)
    # self.k_proj: (H) -> (N_kv * D)
    # self.v_proj: (H) -> (N_kv * D)
    # self.o_proj: (N_q * D) -> (H)

    B, S, _ = hidden_states.shape
    D = self.head_dim
    G = self.num_key_value_groups

    # get the query and kv vectors
    q = self.q_proj(hidden_states).reshape(B, S, -1, D)  # (B, S, N_q, D)
    k = self.k_proj(hidden_states).reshape(B, S, -1, D)  # (B, S, N_kv, D)
    v = self.v_proj(hidden_states).reshape(B, S, -1, D)  # (B, S, N_kv, D)

    cos, sin = position_embeddings

    # reshape cos and sin to be easily expandable later
    cos = cos.reshape(B, S, 1, D)  # (B, S, 1, D)
    sin = sin.reshape(B, S, 1, D)  # (B, S, 1, D)

    def rotate_90(v):
        # rope divides the x and y components of the vectors being rotated into the first and second half, respectively
        x = v[..., :D // 2]
        y = v[..., D // 2:]

        # rotate the components 90 degrees to get an orthogonal vector
        x, y = -y, x

        # put it all back
        return torch.cat([x, y], dim=-1)

    # rotate q
    cos_q = cos.expand_as(q)  # (B, S, N_q, D)
    sin_q = sin.expand_as(q)  # (B, S, N_q, D)
    q = q * cos_q + rotate_90(q) * sin_q

    # rotate k
    cos_k = cos.expand_as(k)  # (B, S, N_kv, D)
    sin_k = sin.expand_as(k)  # (B, S, N_kv, D)
    k = k * cos_k + rotate_90(k) * sin_k

    # the k vectors need to be repeated like:
    # k1 k1 k1 k1 k2 k2 k2 k2 ... kn kn kn kn
    # and this is how I got that
    k = k.reshape(B, S, -1, 1, D)  # (B, S, N_kv, 1, D)
    k = k.expand(-1, -1, -1, G, -1)  # (B, S, N_kv, G, D)
    k = k.reshape(B, S, -1, D)  # (B, S, N_q, D)

    # each seq idx exposes a query vector to the right
    q = q.permute(0, 2, 1, 3)  # (B, N_q, S, D)
    # each seq idx exposes a key vector to the left
    k = k.permute(0, 2, 3, 1)  # (B, N_q, D, S)
    # the vectors dot product together and returns a score for how each seq idx's query responds to each other seq idx's key
    w = q @ k  # (B, N_q, S, S)
    w = w * self.scaling  # sdpa scaling

    if attention_mask is not None:
        attention_mask = attention_mask.expand_as(w)  # (B, N_q, S, S)
    else:
        # by default, seq idx i masks out all seq idxs j > i
        attention_mask = torch.full_like(w, -math.inf)  # (B, N_q, S, S)
        # hence why we get the upper triangular of a matrix of -infs
        attention_mask = torch.triu(attention_mask, diagonal=1)

    # apply attention mask
    w = w + attention_mask

    # same deal here as the k vectors
    v = v.reshape(B, S, -1, 1, D)  # (B, S, N_kv, 1, D)
    v = v.expand(-1, -1, -1, G, -1)  # (B, S, N_kv, G, D)
    v = v.reshape(B, S, -1, D)  # (B, S, N_q, D)

    # scale the values and add them up for each querying seq idx
    attn_weights = torch.softmax(w, dim=-1)
    v = v.permute(0, 2, 1, 3)  # (B, N_q, S, D)
    o = attn_weights @ v  # (B, N_q, S, D)

    # get o into shape to be inputted into o_proj
    o = o.permute(0, 2, 1, 3)  # (B, S, N_q, D)
    o = o.reshape(B, S, -1)  # (B, S, N_q * D)

    # project o and return
    return (
        self.o_proj(o),  # (B, S, H)
        attn_weights if output_attentions else None
    )

LlamaAttention.forward = forward

outputs = model.generate(**inputs, max_new_tokens=20, use_cache=False)
tokenizer.decode(outputs, skip_special_tokens=True)[0]

'What is the capital of France?\nParis, the capital of France, is the largest city in the country and the second-largest'

Holy shit that worked